In [ ]:
"""
TF-IDF + K-Means & HAC Clustering
Input:  complete_preprocessing_final.csv
Output: cluster assignments, evaluation metrics, visualisations
"""

# ============================================================
# 1. Imports
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    adjusted_rand_score,
    normalized_mutual_info_score
)
from sklearn.decomposition import TruncatedSVD
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.sparse import issparse
from collections import Counter

SEED = 42
K = 5


In [ ]:
# ============================================================
# 2. Load Data
# ============================================================
df = pd.read_csv("../complete_preprocessing_final.csv")
df = df.dropna(subset=["trunc_processed_text"]).reset_index(drop=True)
df["trunc_processed_text"] = df["trunc_processed_text"].astype(str)

print(f"Loaded {len(df)} tickets")
print(f"Ticket types: {df['type'].value_counts().to_dict()}")

In [ ]:
# ============================================================
# 3. TF-IDF Vectorisation
# ============================================================
vectorizer = TfidfVectorizer(
    max_features=5000,   # keep top 5000 terms by TF-IDF score
    min_df=5,            # ignore terms appearing in fewer than 5 docs
    max_df=0.85,         # ignore terms appearing in more than 85% of docs
    ngram_range=(1, 2),  # unigrams + bigrams
    sublinear_tf=True    # apply log(tf) scaling to reduce impact of high-freq terms
)

X_tfidf = vectorizer.fit_transform(df["trunc_processed_text"])
print(f"\nTF-IDF matrix shape: {X_tfidf.shape}")
print(f"  → {X_tfidf.shape[0]} documents × {X_tfidf.shape[1]} features")

In [ ]:
# ============================================================
# 4. Dimensionality Reduction for Visualisation (LSA / SVD)
# ============================================================
# Reduce to 2D for scatter plots (does not affect clustering)
svd = TruncatedSVD(n_components=2, random_state=SEED)
X_2d = svd.fit_transform(X_tfidf)

In [ ]:
# ============================================================
# 5. K-Means Clustering  (K=5, seed=42)
# ============================================================
print("\n" + "="*55)
print(" K-MEANS CLUSTERING  (K=5)")
print("="*55)

kmeans = KMeans(n_clusters=K, random_state=SEED, n_init=10, max_iter=300)
km_labels = kmeans.fit_predict(X_tfidf)
df["km_cluster"] = km_labels

km_silhouette = silhouette_score(X_tfidf, km_labels, metric="cosine")
print(f"Silhouette Score (cosine): {km_silhouette:.4f}")

# Cluster size distribution
km_counts = Counter(km_labels)
print("\nCluster sizes:")
for c in sorted(km_counts):
    print(f"  Cluster {c}: {km_counts[c]} tickets")

# Top terms per K-Means cluster
print("\nTop 10 terms per K-Means cluster:")
terms = vectorizer.get_feature_names_out()
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
for i in range(K):
    top_terms = [terms[ind] for ind in order_centroids[i, :10]]
    print(f"  Cluster {i}: {', '.join(top_terms)}")

# ── K-Means scatter plot ──────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
colours = cm.tab10(np.linspace(0, 1, K))
for c in range(K):
    mask = km_labels == c
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               s=5, alpha=0.4, color=colours[c], label=f"Cluster {c}")
ax.set_title("K-Means Clusters (TF-IDF, K=5, LSA 2D projection)")
ax.set_xlabel("LSA Component 1")
ax.set_ylabel("LSA Component 2")
ax.legend(markerscale=3, loc="best")
plt.tight_layout()
plt.show()

# ── Silhouette plot for K-Means ───────────────────────────
sample_sil = silhouette_samples(X_tfidf, km_labels, metric="cosine")
fig, ax = plt.subplots(figsize=(8, 5))
y_lower = 10
for c in range(K):
    c_sil = np.sort(sample_sil[km_labels == c])
    size_c = c_sil.shape[0]
    y_upper = y_lower + size_c
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                     alpha=0.7, color=colours[c])
    ax.text(-0.05, y_lower + 0.5 * size_c, str(c))
    y_lower = y_upper + 10
ax.axvline(x=km_silhouette, color="red", linestyle="--",
           label=f"Avg = {km_silhouette:.3f}")
ax.set_title("Silhouette Plot – K-Means (K=5)")
ax.set_xlabel("Silhouette Coefficient")
ax.set_ylabel("Cluster")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 6. HAC Clustering  (K=5, Ward linkage)
# ============================================================
print("\n" + "="*55)
print(" HAC CLUSTERING  (K=5, Ward linkage)")
print("="*55)

# HAC requires dense input – use LSA-reduced matrix (50 components)
svd50 = TruncatedSVD(n_components=50, random_state=SEED)
X_lsa50 = svd50.fit_transform(X_tfidf)

hac = AgglomerativeClustering(n_clusters=K, linkage="ward")
hac_labels = hac.fit_predict(X_lsa50)
df["hac_cluster"] = hac_labels

hac_silhouette = silhouette_score(X_lsa50, hac_labels, metric="euclidean")
print(f"Silhouette Score (euclidean on LSA-50): {hac_silhouette:.4f}")

# Cluster size distribution
hac_counts = Counter(hac_labels)
print("\nCluster sizes:")
for c in sorted(hac_counts):
    print(f"  Cluster {c}: {hac_counts[c]} tickets")

# Top terms per HAC cluster (via per-cluster mean TF-IDF)
print("\nTop 10 terms per HAC cluster:")
X_dense = X_tfidf.toarray()
for c in range(K):
    mask = hac_labels == c
    centroid = X_dense[mask].mean(axis=0)
    top_idx  = centroid.argsort()[::-1][:10]
    top_terms = [terms[i] for i in top_idx]
    print(f"  Cluster {c}: {', '.join(top_terms)}")

# ── HAC scatter plot ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for c in range(K):
    mask = hac_labels == c
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               s=5, alpha=0.4, color=colours[c], label=f"Cluster {c}")
ax.set_title("HAC Clusters (TF-IDF → LSA-50, Ward, K=5)")
ax.set_xlabel("LSA Component 1")
ax.set_ylabel("LSA Component 2")
ax.legend(markerscale=3, loc="best")
plt.tight_layout()
plt.show()

# ── Dendrogram (sampled for readability) ─────────────────
print("\nBuilding dendrogram (sample of 300 tickets)...")
sample_idx = np.random.RandomState(SEED).choice(len(X_lsa50), 300, replace=False)
Z = linkage(X_lsa50[sample_idx], method="ward")
fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(Z, ax=ax, no_labels=True, color_threshold=0.7*max(Z[:, 2]))
ax.set_title("HAC Dendrogram (Ward linkage, 300-ticket sample)")
ax.set_xlabel("Ticket index")
ax.set_ylabel("Distance")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 7. Alignment with Ground-Truth Labels  (type column)
# ============================================================
print("\n" + "="*55)
print(" ALIGNMENT WITH GROUND-TRUTH  (type column)")
print("="*55)

true_labels = pd.Categorical(df["type"]).codes   # encode string labels → int

km_ari  = adjusted_rand_score(true_labels, km_labels)
km_nmi  = normalized_mutual_info_score(true_labels, km_labels)
hac_ari = adjusted_rand_score(true_labels, hac_labels)
hac_nmi = normalized_mutual_info_score(true_labels, hac_labels)

print(f"\n{'Method':<12} {'ARI':>8} {'NMI':>8} {'Silhouette':>12}")
print("-" * 44)
print(f"{'K-Means':<12} {km_ari:>8.4f} {km_nmi:>8.4f} {km_silhouette:>12.4f}")
print(f"{'HAC':<12} {hac_ari:>8.4f} {hac_nmi:>8.4f} {hac_silhouette:>12.4f}")

# Cross-tabulation: cluster vs ticket type
print("\nK-Means cluster × ticket type:")
print(pd.crosstab(df["km_cluster"], df["type"]))
print("\nHAC cluster × ticket type:")
print(pd.crosstab(df["hac_cluster"], df["type"]))